# 第三十二课:投机解码——小模型代笔,大模型签名

讲义:[32-speculative.md](32-speculative.md)。真训一大一小(d=64/713,600 与 d=32/324,032),贪心验证版投机解码:接受率、前向次数、逐位一致性、诚实的墙钟。

In [ ]:
import math

import time

import numpy as np



# ===== 第 20 课 MiniGPT 原码(含 generate) =====

def softmax(x, axis=-1):

    m = np.max(x, axis=axis, keepdims=True)

    e = np.exp(x - m)

    return e / e.sum(axis=axis, keepdims=True)



class MiniGPT:

    def __init__(self, vocab=16, d=32, n_heads=4, n_layers=2, seed=0):

        rng = np.random.default_rng(seed)

        s = 0.1

        self.E = rng.normal(size=(vocab, d)) * 0.5

        self.dh = d // n_heads

        self.H, self.D, self.NL = n_heads, d, n_layers

        self.layers = [{

            'Wq': rng.normal(size=(d, d))*s, 'Wk': rng.normal(size=(d, d))*s,

            'Wv': rng.normal(size=(d, d))*s, 'Wo': rng.normal(size=(d, d))*s,

            'W1': rng.normal(size=(d, 4*d))*s*0.8, 'W2': rng.normal(size=(4*d, d))*s*0.8,

            'W3': rng.normal(size=(d, 4*d))*s*0.8,

        } for _ in range(n_layers)]

        self.head = rng.normal(size=(d, vocab)) * s



    def rms(self, x, eps=1e-5):

        return x / np.sqrt((x**2).mean(-1, keepdims=True) + eps)



    def rope(self, x, pos):

        half = x.shape[-1] // 2

        ang = pos[:, None] / (10000 ** (np.arange(half)*2/x.shape[-1]))

        c, s = np.cos(ang), np.sin(ang)

        x1, x2 = x[..., :half], x[..., half:]

        return np.concatenate([x1*c - x2*s, x1*s + x2*c], -1)



    def _attn(self, x, pos, W, cache):

        n = x.shape[0]

        q, k, v = self.rope(x@W['Wq'], pos), self.rope(x@W['Wk'], pos), x@W['Wv']

        if cache is not None:

            k, v = np.concatenate([cache[0], k]), np.concatenate([cache[1], v])

        T = k.shape[0]

        mask = np.triu(np.ones((n, T), bool), T - n + 1)

        outs = []

        for h in range(self.H):

            sl = slice(h*self.dh, (h+1)*self.dh)

            s = q[:, sl] @ k[:, sl].T / math.sqrt(self.dh)

            s = np.where(mask, -np.inf, s)

            outs.append(softmax(s) @ v[:, sl])

        return np.concatenate(outs, -1) @ W['Wo'], (k, v)



    def _ffn(self, x, W):

        h = x @ W['W1']

        return (h / (1 + np.exp(-h)) * (x @ W['W3'])) @ W['W2']



    def forward(self, ids, cache=None):

        pos0 = cache[0][0].shape[0] if cache else 0

        x = self.E[list(ids)]

        new_cache = []

        for i, W in enumerate(self.layers):

            a, kv = self._attn(x, np.arange(pos0, pos0+len(ids)), W,

                               cache[i] if cache else None)

            x = x + a

            x = x + self._ffn(self.rms(x), W)

            new_cache.append(kv)

        logits = self.rms(x) @ self.head

        assert np.isfinite(logits).all(), 'NaN! 查掩码/精度/ε'

        return logits, new_cache



    def generate(self, ids, n, T=1.0, top_p=1.0, rng=None):

        rng = rng or np.random.default_rng()

        logits, cache = self.forward(ids)

        out = []

        for _ in range(n):

            p = softmax(logits[-1] / T)

            if top_p < 1.0:

                order = np.argsort(-p); cum = np.cumsum(p[order])

                keep = order[:np.searchsorted(cum, top_p)+1]

                m = np.zeros_like(p); m[keep] = p[keep]; p = m/m.sum()

            nxt = int(rng.choice(len(p), p=p/p.sum()))

            out.append(nxt)

            logits, cache = self.forward([nxt], cache)

        return list(ids) + out

In [ ]:
# ===== 第 23 课反向 + 本课 mask 扩展(合并为一个函数;mask=None 即 23 课原版) =====

def rms_backward(dy, x, eps=1e-5):

    r = np.sqrt((x**2).mean(-1, keepdims=True) + eps)

    y = x / r

    return (dy - y * (dy * y).mean(-1, keepdims=True)) / r



def rope_backward(dy, pos, dim):

    half = dim // 2

    ang = pos[:, None] / (10000 ** (np.arange(half)*2/dim))

    c, s = np.cos(ang), np.sin(ang)

    dy1, dy2 = dy[..., :half], dy[..., half:]

    return np.concatenate([dy1*c + dy2*s, -dy1*s + dy2*c], -1)



def silu_grad(u):

    sig = 1 / (1 + np.exp(-u))

    return sig * (1 + u * (1 - sig))



def loss_and_grads(gpt, ids, targets, mask=None):

    """23 课整机反向 + 25 课可选 mask(监督位加权)。mask=None 等价 23 课原版。"""

    T = len(ids)

    if mask is None:

        mask = [1.0] * T

    marr = np.array(mask, dtype=float)

    Ns = marr.sum()

    pos = np.arange(T)

    tape = []

    x = gpt.E[list(ids)]

    for W in gpt.layers:

        q, k, v = gpt.rope(x@W['Wq'], pos), gpt.rope(x@W['Wk'], pos), x@W['Wv']

        m = np.triu(np.ones((T, T), bool), 1)

        ps, outs = [], []

        for h in range(gpt.H):

            sl = slice(h*gpt.dh, (h+1)*gpt.dh)

            s = q[:, sl] @ k[:, sl].T / math.sqrt(gpt.dh)

            s = np.where(m, -np.inf, s)

            p = softmax(s)

            ps.append(p); outs.append(p @ v[:, sl])

        co = np.concatenate(outs, -1)

        xm = x + co @ W['Wo']

        xr = gpt.rms(xm)

        u, zv = xr @ W['W1'], xr @ W['W3']

        gate = u/(1+np.exp(-u))*zv

        tape.append(dict(x_in=x, xm=xm, xr=xr, u=u, zv=zv, gate=gate, co=co, q=q, k=k, v=v, ps=ps))

        x = xm + gate @ W['W2']                 # 注意:x 入 tape 在覆盖之前(25 课战争故事②)



    xf = x

    logits = gpt.rms(xf) @ gpt.head

    z = logits - logits.max(-1, keepdims=True)

    logp = z - np.log(np.exp(z).sum(-1, keepdims=True))

    loss = -(logp[np.arange(T), np.array(targets)] * marr).sum() / Ns



    grads = {'E': np.zeros_like(gpt.E), 'head': np.zeros_like(gpt.head),

             'layers': [dict.fromkeys(W) for W in gpt.layers]}

    p = softmax(logits)

    onehot = np.zeros_like(p)

    onehot[np.arange(T), np.array(targets)] = 1.0

    dlogits = (p - onehot) * (marr / Ns)[:, None]      # ← mask 版与 23 课的核心差别

    grads['head'] = gpt.rms(xf).T @ dlogits

    dx = rms_backward(dlogits @ gpt.head.T, xf)

    for i in reversed(range(gpt.NL)):

        W, t = gpt.layers[i], tape[i]

        dg = dx @ W['W2'].T

        du = dg * silu_grad(t['u']) * t['zv']

        dzv = dg * (t['u']/(1+np.exp(-t['u'])))

        grads['layers'][i]['W2'] = t['gate'].T @ dx

        grads['layers'][i]['W1'] = t['xr'].T @ du

        grads['layers'][i]['W3'] = t['xr'].T @ dzv

        dxm = dx + rms_backward(du @ W['W1'].T + dzv @ W['W3'].T, t['xm'])

        dc = dxm @ W['Wo'].T

        grads['layers'][i]['Wo'] = t['co'].T @ dxm

        dvs, dqs, dks = [], [], []

        for h in range(gpt.H):

            sl = slice(h*gpt.dh, (h+1)*gpt.dh)

            p_h, v_h, q_h, k_h = t['ps'][h], t['v'][:, sl], t['q'][:, sl], t['k'][:, sl]

            dO = dc[:, sl]

            dvs.append(p_h.T @ dO)

            dp = dO @ v_h.T

            ds = p_h * (dp - (dp*p_h).sum(-1, keepdims=True))

            dqs.append(ds @ k_h / math.sqrt(gpt.dh))

            dks.append(ds.T @ q_h / math.sqrt(gpt.dh))

        dv = np.concatenate(dvs, -1)

        dq = rope_backward(np.concatenate(dqs, -1), pos, gpt.D)

        dk = rope_backward(np.concatenate(dks, -1), pos, gpt.D)

        grads['layers'][i]['Wv'] = t['x_in'].T @ dv

        grads['layers'][i]['Wq'] = t['x_in'].T @ dq

        grads['layers'][i]['Wk'] = t['x_in'].T @ dk

        dx = dxm + dq @ W['Wq'].T + dk @ W['Wk'].T + dv @ W['Wv'].T

    np.add.at(grads['E'], np.array(ids), dx)

    return loss, grads



def flat_params(gpt):

    ps = [gpt.E, gpt.head]

    for W in gpt.layers:

        ps += [W[k] for k in ('Wq', 'Wk', 'Wv', 'Wo', 'W1', 'W2', 'W3')]

    return ps



def flat_grads(grads):

    gs = [grads['E'], grads['head']]

    for L in grads['layers']:

        gs += [L[k] for k in ('Wq', 'Wk', 'Wv', 'Wo', 'W1', 'W2', 'W3')]

    return gs



def clip_norms(gs, max_norm=1.0):

    total = math.sqrt(sum((g ** 2).sum() for g in gs))

    if total > max_norm:

        s = max_norm / total

        for g in gs:

            g *= s

    return total



class MultiAdamW:

    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, wd=0.01):

        self.ps, self.lr, self.b1, self.b2, self.eps, self.wd = params, lr, *betas, eps, wd

        self.m = [np.zeros_like(p) for p in params]

        self.v = [np.zeros_like(p) for p in params]

        self.t = 0

    def step(self, gs):

        self.t += 1

        for p, g, m, v in zip(self.ps, gs, self.m, self.v):

            m *= self.b1; m += (1 - self.b1) * g

            v *= self.b2; v += (1 - self.b2) * g * g

            p -= self.lr * (m/(1-self.b1**self.t)/(np.sqrt(v/(1-self.b2**self.t))+self.eps) + self.wd*p)

In [ ]:
# ===== 训练器(24 课同款) =====
import math
import time

text = open("data/xiyouji.txt", encoding="utf-8").read()
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
V = len(chars)
data = np.array([stoi[c] for c in text], dtype=np.int32)
train_data = data[:int(len(data) * 0.99)]

def flat_params(gpt):
    ps = [gpt.E, gpt.head]
    for W in gpt.layers:
        ps += [W[k] for k in ('Wq', 'Wk', 'Wv', 'Wo', 'W1', 'W2', 'W3')]
    return ps

def flat_grads(grads):
    gs = [grads['E'], grads['head']]
    for L in grads['layers']:
        gs += [L[k] for k in ('Wq', 'Wk', 'Wv', 'Wo', 'W1', 'W2', 'W3')]
    return gs

def clip_norms(gs, max_norm=1.0):
    total = math.sqrt(sum((g ** 2).sum() for g in gs))
    if total > max_norm:
        s = max_norm / total
        for g in gs:
            g *= s
    return total

class MultiAdamW:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, wd=0.01):
        self.ps, self.lr, self.b1, self.b2, self.eps, self.wd = params, lr, *betas, eps, wd
        self.m = [np.zeros_like(p) for p in params]
        self.v = [np.zeros_like(p) for p in params]
        self.t = 0
    def step(self, gs):
        self.t += 1
        for p, g, m, v in zip(self.ps, gs, self.m, self.v):
            m *= self.b1; m += (1 - self.b1) * g
            v *= self.b2; v += (1 - self.b2) * g * g
            p -= self.lr * (m/(1-self.b1**self.t)/(np.sqrt(v/(1-self.b2**self.t))+self.eps) + self.wd*p)

def lr_at(step, total, base_lr=1e-3, warmup=100, floor_ratio=0.1):
    if step < warmup:
        return base_lr * (step + 1) / warmup
    p = (step - warmup) / max(1, total - warmup)
    return base_lr * (floor_ratio + (1 - floor_ratio) * 0.5 * (1 + math.cos(math.pi * p)))

def train_model(d, n_heads, steps, seed=0):
    m = MiniGPT(vocab=V, d=d, n_heads=n_heads, n_layers=2, seed=seed)
    params = flat_params(m)
    opt = MultiAdamW(params)
    rng = np.random.default_rng(42)
    T, B = 64, 16
    for step in range(steps):
        opt.lr = lr_at(step, steps)
        off = rng.integers(0, len(train_data) - T - 1, B)
        gsum = None
        for b in range(B):
            xb = train_data[off[b]:off[b] + T + 1]
            loss, grads = loss_and_grads(m, list(xb[:-1]), list(xb[1:]))
            gs = flat_grads(grads)
            if gsum is None:
                gsum = [g / B for g in gs]
            else:
                for a, g in zip(gsum, gs):
                    a += g / B
        clip_norms(gsum, 1.0)
        opt.step(gsum)
    return m

In [ ]:
# ===== 投机解码本体(讲义 §3) =====
def greedy_baseline(model, prompt, n):
    ids = list(prompt)
    for _ in range(n):
        logits, _ = model.forward(ids)
        ids.append(int(np.argmax(logits[-1])))
    return ids

def greedy_speculative(target, draft, prompt, n, k=4):
    """draft 连写 k 个,target 一次前向验证;输出与 target 独自贪心逐位一致。"""
    ids = list(prompt)
    n_target_calls = 0
    accepted = proposed = 0
    while len(ids) - len(prompt) < n:
        d_ids = list(ids)                       # ---- draft 连写 ----
        draft_tokens = []
        for _ in range(k):
            dl, _ = draft.forward(d_ids)
            t = int(np.argmax(dl[-1]))
            draft_tokens.append(t)
            d_ids.append(t)
        logits, _ = target.forward(d_ids)       # ---- target 一次验证 ----
        n_target_calls += 1
        new_ids = list(ids)
        n_accept = 0
        for dt in draft_tokens:
            tgt_t = int(np.argmax(logits[len(new_ids) - 1]))
            if tgt_t == dt:
                new_ids.append(dt)
                n_accept += 1
            else:
                new_ids.append(tgt_t)           # 纠正,停在这
                break
        else:
            new_ids.append(int(np.argmax(logits[len(d_ids) - 1])))   # bonus
        accepted += n_accept
        proposed += k
        ids = new_ids
    return ids[:len(prompt) + n], accepted, proposed, n_target_calls

In [ ]:
# ===== 训一大一小(各 2000 步,约 4 分钟) =====
t0 = time.time()
big = train_model(d=64, n_heads=4, steps=2000)
print(f"大模型 d=64 2000 步:{time.time()-t0:.0f}s")
t0 = time.time()
small = train_model(d=32, n_heads=4, steps=2000)
n_big = sum(p.size for p in flat_params(big))
n_small = sum(p.size for p in flat_params(small))
print(f"小模型 d=32 2000 步:{time.time()-t0:.0f}s({n_small:,} 参数 = 大模型 {n_big:,} 的 {n_small/n_big:.0%})")

prompt = [stoi[c] for c in "那猴王"]
N = 120

t0 = time.time()
base_ids = greedy_baseline(big, prompt, N)
t_base = time.time() - t0
t0 = time.time()
spec_ids, acc, prop, calls = greedy_speculative(big, small, prompt, N, k=4)
t_spec = time.time() - t0
print(f"\n逐位一致:{spec_ids == base_ids}")
print(f"接受率 {acc/prop:.0%} | target 前向 {calls} 次(基线 {N} 次)")
print(f"基线 {N/t_base:.0f} tok/s | 投机 {N/t_spec:.0f} tok/s | 加速 {t_base/t_spec:.2f}×")
print("样例:" + "".join(itos[i] for i in spec_ids[3:43]))

print("\n[k 扫描]")
for k in (2, 4, 6):
    t0 = time.time()
    _, a2, p2, c2 = greedy_speculative(big, small, prompt, N, k=k)
    dt = time.time() - t0
    print(f"  k={k}: 接受 {a2/p2:.0%} | 前向 {c2} 次 | {N/dt:.0f} tok/s")

In [ ]:
# ---- 作业现场 ----

# TODO(C2): 采样版拒绝采样规则;玩具分布跑 1 万次对照 p

# TODO(C3): draft 换 d=16,画 alpha-r 加速边界

# TODO(D1): 70B+7B 的 r 与理论加速

# TODO(E1): 接受率随生成位置的曲线
